## Importing Modules

In [1]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from parcels import (
    AdvectionRK4_3D,
    FieldSet,
    JITParticle,
    ParticleSet,
    Variable
)
from operator import attrgetter
from pathlib import Path
import datetime
import numpy as np
import xarray as xr
from matplotlib import colormaps as cm
from matplotlib import pyplot as plt
import cartopy
import sys
import pandas as pd
import hvplot.pandas
import geoviews
import random

In [3]:
file_path = Path("/home/jovyan//my_materials/2024-10_parcels-workshop/notebooks/2024_foram_drift/data/Tierney_2019_SOM1_Atlantic.xlsx")
df = pd.read_excel(file_path)
df

,corename,latitude,longitude,wdepth,sdepth,mgca,d18O,d13C,clean,size_lower,...,s_seasonal,grid_lat,grid_lon,include,species,reference1,reference2,reference_link1,reference_link2,notes
0,HM71-22,69.340,-3.610,1833.0,NaN,1.04,2.44,NaN,0,150.0,...,34.979548,69.5,-3.5,1,pachy,Meland2006,NaN,http://dx.doi.org/10.1029/2005GC001078,NaN,NaN
1,HM57-05,69.140,-13.120,1892.0,NaN,0.93,3.55,NaN,0,150.0,...,34.525365,69.5,-13.5,0,pachy,Meland2006,NaN,http://dx.doi.org/10.1029/2005GC001078,NaN,NaN
2,HM94-42,68.450,-22.300,1339.0,NaN,0.91,3.41,NaN,0,150.0,...,33.754073,68.5,-22.5,0,pachy,Meland2006,NaN,http://dx.doi.org/10.1029/2005GC001078,NaN,NaN
3,HM71-12,68.433,-13.867,1547.0,NaN,0.85,3.49,NaN,0,150.0,...,34.621796,68.5,-13.5,1,pachy,Meland2006,NaN,http://dx.doi.org/10.1029/2005GC001078,NaN,NaN
4,HM71-25,67.983,0.233,2855.0,NaN,0.79,2.29,NaN,0,150.0,...,35.141905,67.5,0.5,1,pachy,Meland2006,NaN,http://dx.doi.org/10.1029/2005GC001078,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,AII107-67GGC,-31.915,-36.205,2587.0,1.5,2.80,-0.62,NaN,1,250.0,...,35.807318,-31.5,-36.5,1,sacculifer,Dekens2002,NaN,http://dx.doi.org/10.1029/2001GC000200,NaN,NaN
737,AII107-65GGC,-32.035,-36.188,2795.0,5.5,2.97,-0.48,NaN,1,250.0,...,35.717562,-32.5,-36.5,1,ruber_w,Dekens2002,NaN,http://dx.doi.org/10.1029/2001GC000200,NaN,NaN
738,AII107-65GGC,-32.035,-36.188,2795.0,5.5,2.57,-0.49,NaN,1,250.0,...,35.730217,-32.5,-36.5,1,sacculifer,Dekens2002,NaN,http://dx.doi.org/10.1029/2001GC000200,NaN,NaN
739,VM24-235,-33.250,-19.550,3722.0,NaN,2.38,NaN,NaN,1,250.0,...,35.683300,-33.5,-19.5,1,ruber_w,Dai2019,NaN,https://doi.org/10.1029/2019GC008526,NaN,NaN


In [4]:
df.hvplot.points(
    x="longitude",
    y="latitude",
    c="size_lower",
    geo=True,
    coastline=True,
)

/opt/conda/lib/python3.11/site-packages/cartopy/io/__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


:Overlay
   .Points.I    :Points   [longitude,latitude]   (size_lower)
   .Coastline.I :Feature   [Longitude,Latitude]

In [5]:
df.hvplot.points(
    x="longitude",
    y="latitude",
    c="species",
    geo=True,
    coastline=True,
)

:Overlay
   .Points.I    :Points   [longitude,latitude]   (species)
   .Coastline.I :Feature   [Longitude,Latitude]

## Foram characteristics

In [6]:
foramkey =	{  "pachy": 1,  "incompta": 2,  "bulloides": 3,  "sacculifer": 4,  "ruber_w": 5,  "ruber_p": 6  }

foramvsink = {"pachy" : {"min" : 0.0035,"max" : 0.016},  "incompta" : {"min" : 0.0035,"max" : 0.016},  "bulloides" : {"min" : 0.0035,"max" : 0.016},
              "sacculifer" : {"min" : 0.0035,"max" : 0.016},  "ruber_w" : {"min" : 0.0035,"max" : 0.016},  "ruber_p" : {"min" : 0.0035,"max" : 0.016}} 

foramhabdepth = {"pachy" : {"min" : 50,"max" : 50},  "incompta" : {"min" : 50,"max" : 50},  "bulloides" : {"min" : 50,"max" : 50},
              "sacculifer" : {"min" : 50,"max" : 50},  "ruber_w" : {"min" : 50,"max" : 50},  "ruber_p" : {"min" : 50,"max" : 50}} 

foramlifetime =  {  "pachy": 30,  "incompta": 30,  "bulloides": 30,  "sacculifer": 30,  "ruber_w": 15,  "ruber_p": 15  }

# Input Parameters

In [7]:
i = 304
rng = np.random.default_rng(seed=i)
# Particle locations
lon_bds = df["longitude"][i] #particle_properties.longitude
lat_bds = df["latitude"][i] #particle_properties.latitude
number_particles = 5000 
radius = 0.005 # degree ~500m
r = radius * np.sqrt(rng.uniform(size=(number_particles)))
theta = rng.uniform(size=(number_particles)) * 2 * np.pi
# (x – xM)² + (y – yM)² = r²
lon = lon_bds + r * np.cos(theta)
lat = lat_bds + r * np.sin(theta)
depth = df["wdepth"][i] * np.ones(number_particles)
# m
vSink = foramvsink[df["species"][i]]["max"] * np.ones(number_particles)# = 1400 m/d ... 300 m/d ...
# m/s
#species = foramkey[df["species"][i]] * np.ones(number_particles)
# 1:pachy 2:incompta 3:bulloides 4:sacculifer 5:ruber_w 6:ruber_p
hab_depth = foramhabdepth[df["species"][i]]["max"] * np.ones(number_particles)
# m
#time = fieldset.U.grid.time[-1] # End of FieldSet, set further down below
# Simulation
runtime_in_days = 182 # foramlifetime[df["species"]][i] + ceil(df["wdepth"][i] / (foramvsink[df["species"][i]]["max"] * 86400))
dt_in_hours = 0.1 #360s
# Set to run backwards in time, change further down below
hash = 5 #random.getrandbits(128) what is this for?
outputfilename = df["corename"][i] + "_" + df["species"][i] + "_" +str(i) + "_" + str("%032x" % hash) + "_" + ".zarr"
outputdt_hours = 12 # 1 h too much?

# Definitions

## Files

In [8]:
data_path = "/home/jovyan/shared_materials/model_data/VIKING20X.L46-KKG36107B/"
mesh_file = "/home/jovyan/shared_materials/model_data/mask/VIKING20X.L46-KKG36107B/1_mesh_mask.nc"

U_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_U.nc"))
V_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_V.nc"))
W_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_W.nc_repaire_depthw_time"))
T_files = sorted(Path(data_path).glob("1_VIKING*_1m_199*_grid_T.nc"))

## Define Fieldset

In [9]:
def fieldset_definitions(
    list_of_filenames_U, list_of_filenames_V, list_of_filenames_W,list_of_filenames_T, mesh_mask
):

    filenames = {
        "U": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_W[0],
            "data": list_of_filenames_U,
        },
        "V": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_W[0],
            "data": list_of_filenames_V,
        },
        "W": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_W[0],
            "data": list_of_filenames_W,
        },
        "T": {
            "lon": (mesh_mask),
            "lat": (mesh_mask),
            "depth": list_of_filenames_T[0],
            "data": list_of_filenames_T,
        },
    }

    variables = {
        "U": "vozocrtx",
        "V": "vomecrty",
        "W": "vovecrtz",
        "T": "votemper",
    }

    dimensions = {
        "U": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "depthw",
            "time": "time_counter",
        },
        "V": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "depthw",
            "time": "time_counter",
        },
        "W": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "depthw",
            "time": "time_counter",
        },
        "T": {
            "lon": "glamf",
            "lat": "gphif",
            "depth": "deptht",
            "time": "time_counter",
        },
    }

    return FieldSet.from_nemo(
        filenames,
        variables,
        dimensions,
        # indices=indices,
        mesh="spherical",
        tracer_interp_method="cgrid_tracer",
        allow_time_extrapolation=False,
    )

fieldset = fieldset_definitions(
    list_of_filenames_U=U_files,
    list_of_filenames_V=V_files,
    list_of_filenames_W=W_files,
    list_of_filenames_T=T_files,
    mesh_mask=mesh_file,
)

## Define Particle Set

In [10]:
# Time
time = np.mean(fieldset.U.grid.time)
#time = fieldset.U.grid.time[-1] # end of fieldset data

In [11]:
extra_vars = [
    Variable("temperature", dtype=np.float32, initial= -999
    ),
    Variable("sinkingSpeed", dtype=np.float32
    ),
    Variable("habDepth", dtype=np.float32
    ),
]

In [12]:
# Records Temperature and Distance
TSampler = JITParticle.add_variables(extra_vars)

In [13]:
pset = ParticleSet.from_list(
    fieldset = fieldset,
    pclass = TSampler,
    lon = lon,
    lat = lat,
    depth = depth,
    time = time * np.ones(np.size(lon)),
    repeatdt = datetime.timedelta(days=1),
    sinkingSpeed = vSink,
    habDepth = hab_depth
)

In [14]:
#pset

## Kernels

In [15]:
def CheckError(particle, fieldset, time):
    if particle.state >= 50:  # This captures all Errors
        particle.delete()

In [16]:
def SampleT(particle, fieldset, time): # Interpolates particle temperature from Fieldset temperature
    particle.temperature = fieldset.T[time, particle.depth, particle.lat, particle.lon]

In [17]:
def ActiveSinking(particle,fieldset,time):
    if particle.depth <= particle.habDepth:
        particle.sinkingSpeed = 0
        #particle.dt =  3600 # s
    particle_ddepth += particle.sinkingSpeed * particle.dt

## Output File

In [18]:
outputfile = pset.ParticleFile(
    name=outputfilename, 
    outputdt=datetime.timedelta(hours=outputdt_hours),
    chunks=(10000,48)
)
outputfile.metadata["species"] = str(df["species"][i])
outputfile.metadata["vsink"] = str(foramvsink[df["species"][i]]["max"])
outputfile.metadata["habdepth"] = str(hab_depth[0])

## EXECUTE KERNELS

pset.execute(
    [SampleT,AdvectionRK4_3D,ActiveSinking,CheckError],
    runtime= datetime.timedelta(days=runtime_in_days),
    dt= -datetime.timedelta(hours=dt_in_hours), # note the minus
    output_file =  outputfile
)

In [19]:
ds = xr.open_zarr(outputfilename).compute()

# Results

In [20]:
ds

<xarray.Dataset> Size: 15GB
Dimensions:       (trajectory: 910000, obs: 384)
Coordinates:
  * obs           (obs) int32 2kB 0 1 2 3 4 5 6 ... 377 378 379 380 381 382 383
  * trajectory    (trajectory) int64 7MB 0 1 2 3 ... 909996 909997 909998 909999
Data variables:
    habDepth      (trajectory, obs) float32 1GB 50.0 50.0 50.0 ... nan nan nan
    lat           (trajectory, obs) float64 3GB 31.9 31.91 31.93 ... nan nan nan
    lon           (trajectory, obs) float64 3GB -64.3 -64.29 -64.28 ... nan nan
    sinkingSpeed  (trajectory, obs) float32 1GB 0.016 0.016 0.016 ... nan nan
    temperature   (trajectory, obs) float32 1GB 1.774 2.08 2.856 ... nan nan nan
    time          (trajectory, obs) datetime64[ns] 3GB 1994-12-31T09:12:00 .....
    z             (trajectory, obs) float64 3GB 4.469e+03 3.782e+03 ... nan nan
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    habdepth:               50.0
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        NewParticleSampleTAdvectionRK4_3DActiveSinkingChe...
    parcels_mesh:           spherical
    parcels_version:        3.0.6
    species:                sacculifer
    vsink:                  0.016